<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%B7%D0%B0_2_%D0%BD%D0%B5%D0%B4_%D0%9F%D0%BE%D1%85_%D1%8E%D0%B7%D0%B5%D1%80%D1%8B_%D0%91%D0%B5%D0%B7_%D0%92%D0%B7%D0%B0%D0%B8%D0%BC_%D0%9E%D0%B3%D1%80_100_ml_ozon_recsys_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OZON RecSys Baseline - Рекомендательная система для категории Apparel
Этот ноутбук содержит базовое решение для задачи предсказания следующей покупки пользователя в категории одежды, обуви и аксессуаров.

## Задача
- Предсказать топ-100 товаров для каждого пользователя из тестовой выборки
- Метрика оценки: NDCG@100
- Данные: ~38GB в формате parquet, 1.6B взаимодействий, 19M заказов


In [1]:
# === 1. Монтируем Google Drive, задаём пути к данным (структура Colab/Яндекс) ===
from google.colab import drive
drive.mount('/content/drive')

# Шаг 1. Импорты и пути (Colab/локально, без лишних библиотек)
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict, Counter, deque # Добавлены для подсчета популярности
import glob

# Пути к данным (замените на свои)
ORDERS_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_orders_data/final_apparel_orders_data_07'
ORDERS_PATH2 = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/raw/ml_ozon_recsys_train_final_apparel_orders_data'
TEST_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/ml_ozon_recsys_test'

# Шаг 2. Загрузка всех заказов (train)
def load_orders():
    print("Загружаем тренировочные данные заказов...")
    orders = []
    for path in [ORDERS_PATH, ORDERS_PATH2]:
        for f in Path(path).rglob('*.parquet'):
            orders.append(pd.read_parquet(f))
    df = pd.concat(orders, ignore_index=True)
    # Обогащаем created_date, если нужно
    if 'created_date' in df.columns and df['created_date'].isna().sum():
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_date'].fillna(df['created_timestamp'].dt.date)
        df['created_date'] = pd.to_datetime(df['created_date'])
    elif 'created_date' not in df.columns and 'created_timestamp' in df.columns:
        # Если created_date вообще отсутствует, создаем её
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_timestamp'].dt.date
        df['created_date'] = pd.to_datetime(df['created_date'])
    print(f"Загружено заказов: {len(df):,}")
    return df

orders_df = load_orders()

Mounted at /content/drive
Загружаем тренировочные данные заказов...
Загружено заказов: 20,362,338


In [2]:
print("=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===")
print("\n👉 Первые 2 строки с типами данных:")
print(orders_df.head(2).to_markdown(tablefmt="grid"))

print("\n📊 Схема данных:")
for col in orders_df.columns:
    print(f"  • {col}: {orders_df[col].dtype}")

=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===

👉 Первые 2 строки с типами данных:
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|    |   item_id |   user_id | created_timestamp          | last_status      | last_status_timestamp   | created_date        |
+====+===========+===========+============================+==================+=========================+=====================+
|  0 | 332361399 |      2841 | 2025-07-14 08:38:22.250000 | proccesed_orders | 2025-07-14 11:35:22     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|  1 | 153141296 |      3681 | 2025-07-14 09:13:56.410000 | proccesed_orders | 2025-07-14 10:47:53     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+

📊 Схема данных:
  • ite

In [3]:
# Шаг 3. Загрузка тестовых пользователей
def load_test_users():
    print("Загружаем тестовых пользователей...")
    test_files = glob.glob(f'{TEST_PATH}/*.parquet')
    users = set()
    for f in tqdm(test_files, desc="Обработка тестовых файлов"):
        df_part = pd.read_parquet(f)
        if 'user_id' in df_part.columns:
            users.update(df_part['user_id'].unique())
    print(f"Найдено уникальных тестовых пользователей: {len(users):,}")
    return list(users)

test_users = load_test_users()

Загружаем тестовых пользователей...


Обработка тестовых файлов: 100%|██████████| 1/1 [00:01<00:00,  1.09s/it]

Найдено уникальных тестовых пользователей: 470,347


In [4]:
print("=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
test_users_df = pd.DataFrame({'user_id': test_users})
print(f"📊 Размер: {len(test_users_df):,} строк")
print("\n👉 Первые 2 строки:")
print(test_users_df.head(2).to_string())
print("\n📋 Колонки и типы данных:")
for col in test_users_df.columns:
    print(f"  • {col:20} {test_users_df[col].dtype}")

=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
📊 Размер: 470,347 строк

👉 Первые 2 строки:
   user_id
0        1
1  3145730

📋 Колонки и типы данных:
  • user_id              int32


In [5]:
print("\n\nАНАЛИЗ ЗАКАЗОВ")
print("=" * 50)
print(f"Общее количество заказов: {len(orders_df):,}")
print(f"Уникальных пользователей: {orders_df['user_id'].nunique():,}")
print(f"Уникальных товаров: {orders_df['item_id'].nunique():,}")

if 'created_date' in orders_df.columns:
    min_date = orders_df['created_date'].min().date()
    max_date = orders_df['created_date'].max().date()
    print(f"Период данных: {min_date} - {max_date}")

if 'last_status' in orders_df.columns:
    print("\nРаспределение статусов заказов:")
    status_counts = orders_df['last_status'].value_counts()
    status_counts_pct = orders_df['last_status'].value_counts(normalize=True) * 100
    for status, count in status_counts.items():
        pct = status_counts_pct[status]
        print(f"  {status}: {count:,} ({pct:.1f}%)")




АНАЛИЗ ЗАКАЗОВ
Общее количество заказов: 20,362,338
Уникальных пользователей: 842,254
Уникальных товаров: 4,679,218
Период данных: 2025-01-01 - 2025-07-15

Распределение статусов заказов:
  delivered_orders: 10,420,894 (51.2%)
  canceled_orders: 8,420,631 (41.4%)
  proccesed_orders: 1,520,813 (7.5%)


In [6]:
print("\nТоп-10 самых популярных товаров (по количеству заказов):")
top_items_orders = orders_df['item_id'].value_counts().head(10)
for item_id, count in top_items_orders.items():
    print(f"  Товар {item_id}: {count:,} заказов")



Топ-10 самых популярных товаров (по количеству заказов):
  Товар 51974017: 13,361 заказов
  Товар 187052809: 12,384 заказов
  Товар 207631139: 8,877 заказов
  Товар 143497612: 4,096 заказов
  Товар 119105606: 3,497 заказов
  Товар 247423473: 3,188 заказов
  Товар 77696741: 2,735 заказов
  Товар 175287070: 2,725 заказов
  Товар 285009143: 2,634 заказов
  Товар 201930716: 2,624 заказов


In [7]:
# Шаг 4. Разделение на тренировочную и тестовую выборки по времени
print("=== 📊 РАЗДЕЛЕНИЕ НА TRAIN/TEST ПО ВРЕМЕНИ ===")
train_end_date = pd.to_datetime('2025-07-08')  # Первая неделя: 2025-07-02 до 2025-07-08
test_start_date = pd.to_datetime('2025-07-09')  # Вторая неделя: 2025-07-09 до 2025-07-15

# Тренировочные данные (только доставленные заказы за первую неделю)
train_orders = orders_df[
    (orders_df['last_status'] == 'delivered_orders') &
    (orders_df['created_date'] >= pd.to_datetime('2025-07-02')) &
    (orders_df['created_date'] <= train_end_date)
].copy()

# Тестовые данные (заказы за вторую неделю - для оценки)
test_orders = orders_df[
    (orders_df['last_status'] == 'delivered_orders') &
    (orders_df['created_date'] >= test_start_date) &
    (orders_df['created_date'] <= pd.to_datetime('2025-07-15'))
].copy()

print(f"📅 Тренировочный период: 2025-07-02 - {train_end_date.date()}")
print(f"📅 Тестовый период: {test_start_date.date()} - 2025-07-15")
print(f"📦 Тренировочных заказов: {len(train_orders):,}")
print(f"📦 Тестовых заказов: {len(test_orders):,}")

=== 📊 РАЗДЕЛЕНИЕ НА TRAIN/TEST ПО ВРЕМЕНИ ===
📅 Тренировочный период: 2025-07-02 - 2025-07-08
📅 Тестовый период: 2025-07-09 - 2025-07-15
📦 Тренировочных заказов: 317,404
📦 Тестовых заказов: 167,241


In [8]:
# Шаг 5. Аналитическая "модель популярности" по тренировочным данным
def get_popular_items(train_orders, top_k=100):
    """Получить топ-K популярных товаров из тренировочной выборки"""
    print("\n=== 🏆 РАСЧЕТ ПОПУЛЯРНЫХ ТОВАРОВ (TRAIN) ===")
    item_counts = train_orders['item_id'].value_counts().head(top_k)
    popular_items = item_counts.index.tolist()

    print(f"Топ-{top_k} популярных товаров рассчитан")
    print("\n📋 Топ-10 самых популярных товаров:")
    for i, (item_id, count) in enumerate(item_counts.head(10).items(), 1):
        print(f"  {i:2d}. Товар {item_id:>12}: {count:>6,} заказов")

    return popular_items

popular_items = get_popular_items(train_orders, top_k=100)


=== 🏆 РАСЧЕТ ПОПУЛЯРНЫХ ТОВАРОВ (TRAIN) ===
Топ-100 популярных товаров рассчитан

📋 Топ-10 самых популярных товаров:
   1. Товар    175287070:    343 заказов
   2. Товар     51974017:    306 заказов
   3. Товар    166327353:    263 заказов
   4. Товар    187052809:    262 заказов
   5. Товар    247423473:    204 заказов
   6. Товар     11083343:    195 заказов
   7. Товар    334086992:    184 заказов
   8. Товар     63987378:    171 заказов
   9. Товар    206494927:    164 заказов
  10. Товар    201930716:    136 заказов


In [9]:
# Шаг 6. Получение тестовых пользователей (те, кто сделал заказы во вторую неделю)
test_users_during_period = test_orders['user_id'].unique().tolist()
print(f"\n=== 👥 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
print(f"Найдено тестовых пользователей: {len(test_users_during_period):,}")


=== 👥 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
Найдено тестовых пользователей: 96,159


In [10]:
# Преобразуем список тестовых пользователей в множество для быстрого поиска
test_users_set = set(test_users_during_period)
print(f"✅ Множество тестовых пользователей создано. Размер: {len(test_users_set):,}")

✅ Множество тестовых пользователей создано. Размер: 96,159


In [11]:
# Шаг 7. Вычисление истории покупок для тестовых пользователей (их покупки в первую неделю)
def build_user_history_first_week(filtered_orders_df, test_users):
    """Построить историю покупок тестовых пользователей из предоставленного датафрейма"""
    print("\n=== 📚 ПОСТРОЕНИЕ ИСТОРИИ ПОКУПОК (из предоставленных данных) ===")

    # Предполагаем, что filtered_orders_df уже содержит нужные заказы (например, train_orders)
    # Фильтруем только по тестовым пользователям
    user_history = filtered_orders_df[
        (filtered_orders_df['user_id'].isin(test_users))
    ].groupby('user_id')['item_id'].apply(set).to_dict()

    # Конвертируем в список для совместимости
    user_preferences = {uid: list(items) for uid, items in user_history.items()}

    print(f"История покупок построена для {len(user_preferences):,} пользователей")
    return user_preferences

# Передаем train_orders вместо orders_df
user_preferences = build_user_history_first_week(train_orders, test_users_during_period)


=== 📚 ПОСТРОЕНИЕ ИСТОРИИ ПОКУПОК (из предоставленных данных) ===
История покупок построена для 40,775 пользователей


In [12]:
# Шаг 8. Генерация рекомендаций
def generate_recommendations(test_users, popular_items, user_preferences, top_k=100):
    """Генерация рекомендаций: популярные товары, исключая уже купленные"""
    print("\n=== 🎯 ГЕНЕРАЦИЯ РЕКОМЕНДАЦИЙ ===")
    recs = {}
    for uid in tqdm(test_users, desc='Генерация рекомендаций'):
        bought = set(user_preferences.get(uid, []))
        recs[uid] = [item for item in popular_items if item not in bought][:top_k]
    print(f"Рекомендации сгенерированы для {len(recs):,} пользователей")
    return recs

recommendations = generate_recommendations(test_users_during_period, popular_items, user_preferences)


=== 🎯 ГЕНЕРАЦИЯ РЕКОМЕНДАЦИЙ ===


Генерация рекомендаций: 100%|██████████| 96159/96159 [00:00<00:00, 114772.04it/s]

Рекомендации сгенерированы для 96,159 пользователей


In [13]:
# Шаг 10. Подготовка тестовых меток (что пользователи действительно купили во вторую неделю)
def prepare_ground_truth(test_orders):
    """Подготовить ground truth: что пользователи купили в тестовый период"""
    print("\n=== ✅ ПОДГОТОВКА GROUND TRUTH ===")
    ground_truth = test_orders.groupby('user_id')['item_id'].apply(set).to_dict()

    print(f"Ground truth подготовлен для {len(ground_truth):,} пользователей")

    # Показываем примеры для проверки
    if ground_truth:
        # Берем первых 3 пользователя для примера
        sample_users = sorted(list(ground_truth.keys()))[:3]  # Сортируем для воспроизводимости
        print("\n🔍 Примеры ground truth для проверки:")
        for i, user_id in enumerate(sample_users, 1):
            items = list(ground_truth[user_id])  # Все товары пользователя
            print(f"  Пользователь {user_id}: {items} ({len(ground_truth[user_id])} всего покупок)")

            # Дополнительная проверка - показываем те же данные из исходного тестового датафрейма
            user_test_data = test_orders[test_orders['user_id'] == user_id]
            test_items = user_test_data['item_id'].tolist()
            print(f"  Пользователь {user_id} (из теста): {test_items} ({len(test_items)} записей в тесте)")

    return ground_truth

ground_truth = prepare_ground_truth(test_orders)


=== ✅ ПОДГОТОВКА GROUND TRUTH ===
Ground truth подготовлен для 96,159 пользователей

🔍 Примеры ground truth для проверки:
  Пользователь 60: [18766015, 111169159] (2 всего покупок)
  Пользователь 60 (из теста): [111169159, 18766015] (2 записей в тесте)
  Пользователь 91: [27861200, 15428515, 27733496, 104094923] (4 всего покупок)
  Пользователь 91 (из теста): [15428515, 27861200, 27733496, 104094923] (4 записей в тесте)
  Пользователь 111: [179543784, 83982210] (2 всего покупок)
  Пользователь 111 (из теста): [179543784, 83982210] (2 записей в тесте)


In [14]:
# Шаг 10. Расчет NDCG@100
def ndcg_at_k(y_true, y_pred, k=100):
    """
    Рассчитать NDCG@k для одного пользователя
    y_true: множество реально купленных товаров
    y_pred: список рекомендованных товаров
    """
    if not y_true:
        return 0.0

    # Бинарная оценка: 1 если товар куплен, 0 если нет
    dcg = 0.0
    for i, item in enumerate(y_pred[:k]):
        if item in y_true:
            dcg += 1.0 / np.log2(i + 2)  # log2(1+pos) = log2(pos+2) т.к. индекс с 0

    # IDCG - идеальный DCG (все релевантные товары в начале)
    idcg = 0.0
    ideal_len = min(len(y_true), k)
    for i in range(ideal_len):
        idcg += 1.0 / np.log2(i + 2)

    if idcg == 0:
        return 0.0

    return dcg / idcg

def calculate_ndcg_batch(recommendations, ground_truth, k=100):
    """Рассчитать NDCG@k для всех пользователей"""
    print(f"\n=== 📊 РАСЧЕТ МЕТРИКИ NDCG@{k} ===")

    ndcg_scores = []
    users_evaluated = 0

    for uid in tqdm(recommendations.keys(), desc='Расчет NDCG'):
        if uid in ground_truth:
            y_pred = recommendations[uid]
            y_true = ground_truth[uid]

            ndcg = ndcg_at_k(y_true, y_pred, k)
            ndcg_scores.append(ndcg)
            users_evaluated += 1

    mean_ndcg = np.mean(ndcg_scores) if ndcg_scores else 0.0
    print(f"Оценено пользователей: {users_evaluated:,}")
    print(f"NDCG@{k}: {mean_ndcg:.6f}")

    return mean_ndcg

# Расчет метрики
ndcg_score = calculate_ndcg_batch(recommendations, ground_truth, k=100)


=== 📊 РАСЧЕТ МЕТРИКИ NDCG@100 ===


Расчет NDCG: 100%|██████████| 96159/96159 [00:01<00:00, 91579.40it/s]

Оценено пользователей: 96,159
NDCG@100: 0.007641


In [15]:

# Шаг 1: Создаем бинарную матрицу взаимодействий "user-item" для тренировочных данных
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import numpy as np

print("Создаем бинарную матрицу user-item...")

# Получаем всех уникальных пользователей и товаров из train_orders
train_users = train_orders['user_id'].unique()
train_items = train_orders['item_id'].unique()

# Создаем маппинги для быстрого доступа
user_to_idx = {user: idx for idx, user in enumerate(train_users)}
item_to_idx = {item: idx for idx, item in enumerate(train_items)}

# Создаем разреженную бинарную матрицу
rows, cols, data = [], [], []

for _, row in tqdm(train_orders.iterrows(), total=len(train_orders), desc="Построение матрицы"):
    user_idx = user_to_idx[row['user_id']]
    item_idx = item_to_idx[row['item_id']]
    rows.append(user_idx)
    cols.append(item_idx)
    data.append(1)  # Бинарная метка: 1 = покупка

# Создаем CSR матрицу
user_item_matrix = csr_matrix((data, (rows, cols)),
                             shape=(len(train_users), len(train_items)))

print(f"Матрица создана: {user_item_matrix.shape[0]} users × {user_item_matrix.shape[1]} items")
print(f"Плотность матрицы: {(user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]) * 100):.6f}%")

Создаем бинарную матрицу user-item...


Построение матрицы: 100%|██████████| 317404/317404 [00:16<00:00, 18747.47it/s]


Матрица создана: 162264 users × 204901 items
Плотность матрицы: 0.000951%


In [16]:
# Шаг 2: Вычисляем косинусную схожесть между пользователями
print("Вычисляем матрицу схожести пользователей...")
user_similarity = cosine_similarity(user_item_matrix)
print(f"Матрица схожести: {user_similarity.shape}")

Вычисляем матрицу схожести пользователей...
Матрица схожести: (162264, 162264)


In [17]:
# Шаг 3: Функция для поиска похожих пользователей
def find_similar_users(test_user_id, top_n=10):
    """
    Найти top-N похожих пользователей для тестового пользователя
    """
    if test_user_id not in user_to_idx:
        return []  # Пользователь не найден в тренировочных данных

    test_user_idx = user_to_idx[test_user_id]
    similarities = user_similarity[test_user_idx]

    # Получаем индексы самых похожих пользователей (исключая самого себя)
    similar_indices = np.argsort(similarities)[::-1][1:top_n+1]

    # Преобразуем обратно в user_id
    similar_users = []
    for idx in similar_indices:
        if similarities[idx] > 0:  # Только пользователи с положительной схожестью
            similar_user_id = train_users[idx]
            similar_users.append((similar_user_id, similarities[idx]))

    return similar_users


In [18]:
# Шаг 4: Пример использования для нескольких тестовых пользователей
print("\n=== 🔍 ПРИМЕРЫ ПОИСКА ПОХОЖИХ ПОЛЬЗОВАТЕЛЕЙ ===")

# Берем первых 5 тестовых пользователей, которые есть в тренировочных данных
test_users_in_train = [uid for uid in test_users_during_period[:20] if uid in user_to_idx]

for i, test_user in enumerate(test_users_in_train[:5]):
    similar_users = find_similar_users(test_user, top_n=5)
    print(f"\n👤 Тестовый пользователь {test_user}:")
    if similar_users:
        for j, (similar_user, similarity) in enumerate(similar_users, 1):
            print(f"   {j}. Пользователь {similar_user} (схожесть: {similarity:.4f})")
    else:
        print("   Нет похожих пользователей в тренировочных данных")



=== 🔍 ПРИМЕРЫ ПОИСКА ПОХОЖИХ ПОЛЬЗОВАТЕЛЕЙ ===

👤 Тестовый пользователь 12390:
   1. Пользователь 1837590 (схожесть: 0.5000)
   2. Пользователь 3586921 (схожесть: 0.5000)
   3. Пользователь 4655561 (схожесть: 0.5000)
   4. Пользователь 3302001 (схожесть: 0.5000)
   5. Пользователь 1853971 (схожесть: 0.5000)

👤 Тестовый пользователь 462911:
   1. Пользователь 3373580 (схожесть: 0.5774)
   2. Пользователь 3159511 (схожесть: 0.5774)
   3. Пользователь 2320920 (схожесть: 0.5774)
   4. Пользователь 2316030 (схожесть: 0.5774)
   5. Пользователь 4968600 (схожесть: 0.5774)

👤 Тестовый пользователь 935441:
   1. Пользователь 955280 (схожесть: 0.3536)
   2. Пользователь 1722310 (схожесть: 0.2887)
   3. Пользователь 1719051 (схожесть: 0.2236)
   4. Пользователь 1152290 (схожесть: 0.2236)
   5. Пользователь 3001131 (схожесть: 0.2236)

👤 Тестовый пользователь 946081:
   1. Пользователь 594340 (схожесть: 0.3333)
   2. Пользователь 3189420 (схожесть: 0.3333)
   3. Пользователь 3069691 (схожесть: 0.3

In [19]:
# Шаг 5: Полный поиск для всех тестовых пользователей (без ограничения)
print(f"\n=== 🚀 ПОЛНЫЙ ПОИСК ДЛЯ ВСЕХ ТЕСТОВЫХ ПОЛЬЗОВАТЕЛЕЙ ===")

# Создаем словарь для хранения похожих пользователей
similar_users_dict = {}

test_users_to_process = [uid for uid in test_users_during_period if uid in user_to_idx]
print(f"Найдено {len(test_users_to_process)} тестовых пользователей в тренировочных данных")

# Убираем ограничение [:1000] - обрабатываем всех
for test_user in tqdm(test_users_to_process, desc="Поиск похожих пользователей"):
    similar_users_dict[test_user] = find_similar_users(test_user, top_n=10)

print(f"Поиск завершен! Найдены похожие пользователи для {len(similar_users_dict)} тестовых пользователей")


=== 🚀 ПОЛНЫЙ ПОИСК ДЛЯ ВСЕХ ТЕСТОВЫХ ПОЛЬЗОВАТЕЛЕЙ ===
Найдено 40775 тестовых пользователей в тренировочных данных


Поиск похожих пользователей: 100%|██████████| 40775/40775 [08:24<00:00, 80.85it/s]

Поиск завершен! Найдены похожие пользователи для 40775 тестовых пользователей


In [20]:
# Шаг 6: Пример статистики для полного поиска
similarity_counts = [len(similar) for similar in similar_users_dict.values()]
print(f"\n📊 Статистика полного поиска:")
print(f"Среднее количество похожих пользователей: {np.mean(similarity_counts):.2f}")
print(f"Максимальное количество: {max(similarity_counts)}")
print(f"Минимальное количество: {min(similarity_counts)}")
print(f"Пользователей без похожих: {sum(1 for x in similarity_counts if x == 0)}")

# Дополнительная статистика
non_zero_similarities = [count for count in similarity_counts if count > 0]
if non_zero_similarities:
    print(f"Среднее количество похожих (только с похожими): {np.mean(non_zero_similarities):.2f}")


📊 Статистика полного поиска:
Среднее количество похожих пользователей: 3.40
Максимальное количество: 10
Минимальное количество: 0
Пользователей без похожих: 13510
Среднее количество похожих (только с похожими): 5.09


In [21]:
print("\n1. Словарь похожих пользователей (similar_users_dict):")
print(f"Тип: {type(similar_users_dict)}")
print(f"Размер: {len(similar_users_dict)} записей")

# Покажем пример структуры
sample_user_id = list(similar_users_dict.keys())[0] if similar_users_dict else None
if sample_user_id:
    sample_data = similar_users_dict[sample_user_id]
    print(f"Пример для пользователя {sample_user_id}:")
    print(f"  Тип: {type(sample_data)}")
    print(f"  Содержание: {sample_data}")
    if sample_data:
        print(f"  Первый элемент: {sample_data[0]}")
        print(f"  Тип первого элемента: {type(sample_data[0])}")

print("\n" + "="*50)


1. Словарь похожих пользователей (similar_users_dict):
Тип: <class 'dict'>
Размер: 40775 записей
Пример для пользователя 12390:
  Тип: <class 'list'>
  Содержание: [(np.int32(1837590), np.float64(0.5)), (np.int32(3586921), np.float64(0.5)), (np.int32(4655561), np.float64(0.5)), (np.int32(3302001), np.float64(0.5)), (np.int32(1853971), np.float64(0.5)), (np.int32(3231830), np.float64(0.5)), (np.int32(354401), np.float64(0.5)), (np.int32(2271270), np.float64(0.5)), (np.int32(4141101), np.float64(0.5)), (np.int32(629551), np.float64(0.5))]
  Первый элемент: (np.int32(1837590), np.float64(0.5))
  Тип первого элемента: <class 'tuple'>



In [22]:
# Сохранение только похожих пользователей и необходимых данных
import pickle
from datetime import datetime
from pathlib import Path

# Создаем путь для сохранения
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/processed'
Path(SAVE_PATH).mkdir(parents=True, exist_ok=True)

print("=== 💾 СОХРАНЕНИЕ ДАННЫХ ДЛЯ БЫСТРОЙ ЗАГРУЗКИ ===")

# 1. Сохраняем словарь похожих пользователей
similar_users_file = f"{SAVE_PATH}/similar_users_dict.pkl"
with open(similar_users_file, 'wb') as f:
    pickle.dump(similar_users_dict, f)
print(f"✅ Похожие пользователи сохранены: {similar_users_file}")


=== 💾 СОХРАНЕНИЕ ДАННЫХ ДЛЯ БЫСТРОЙ ЗАГРУЗКИ ===
✅ Похожие пользователи сохранены: /content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/processed/similar_users_dict.pkl


In [23]:
import numpy as np
from collections import defaultdict, Counter
from tqdm import tqdm

def generate_recommendations_hybrid(
    test_users,
    popular_items,
    user_preferences,
    similar_users_dict,
    orders_df, # Нам понадобится полный датафрейм для получения покупок похожих пользователей
    top_k=100
):
    """
    Генерация гибридных рекомендаций:
    - Для пользователей с историей и похожими: CF на базе пользователей.
    - Для остальных: популярные товары, исключая уже купленные.
    """
    print("\n=== 🎯 ГЕНЕРАЦИЯ ГИБРИДНЫХ РЕКОМЕНДАЦИЙ ===")

    # 1. Подготовка: создадим словарь покупок для быстрого доступа
    # Это нужно, чтобы быстро получить список покупок для любого пользователя
    print("Подготовка индекса покупок пользователей...")
    user_items_index = orders_df[
        (orders_df['last_status'] == 'delivered_orders')
    ].groupby('user_id')['item_id'].apply(set).to_dict()
    print(f"Индекс покупок создан для {len(user_items_index)} пользователей.")

    recommendations = {}

    for uid in tqdm(test_users, desc='Генерация рекомендаций'):
        rec_items = []

        # Получаем историю текущего пользователя (то, что он уже купил)
        bought_by_user = set(user_preferences.get(uid, []))

        # Получаем список похожих пользователей
        similar_users_list = similar_users_dict.get(uid, [])

        # --- Логика гибридной модели ---
        if uid in user_preferences and similar_users_list:
            # --- Сценарий 1: Есть история и есть похожие пользователи ---
            # Используем Collaborative Filtering

            # Счетчик для товаров, купленных похожими пользователями
            # Ключ: item_id, Значение: сумма "весов" (схожестей) пользователей, купивших его
            item_scores = defaultdict(float)

            # Проходим по каждому похожему пользователю
            for similar_user_id, similarity_score in similar_users_list:
                # Получаем товары, купленные похожим пользователем
                items_bought_by_similar = user_items_index.get(similar_user_id, set())

                # Добавляем эти товары в счетчик, взвешивая по степени схожести
                for item in items_bought_by_similar:
                    # Убедимся, что товар еще не куплен целевым пользователем
                    if item not in bought_by_user:
                         # Увеличиваем "счет" товара на "силу" схожести похожего пользователя
                        item_scores[item] += similarity_score

            # Сортируем товары по убыванию "счета" (релевантности)
            # item_scores.items() дает пары (item_id, score)
            sorted_items = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)

            # Извлекаем только ID товаров, отсекаем по top_k
            rec_items = [item_id for item_id, score in sorted_items[:top_k]]

            # Дополнительная проверка: если рекомендаций меньше top_k,
            # дополним популярными товарами (исключая уже купленные и уже добавленные)
            if len(rec_items) < top_k:
                # Множество уже рекомендованных товаров, чтобы не дублировать
                rec_items_set = set(rec_items)
                # Добавляем популярные товары, пока не наберем top_k
                for pop_item in popular_items:
                     if len(rec_items) >= top_k:
                        break
                     if pop_item not in bought_by_user and pop_item not in rec_items_set:
                         rec_items.append(pop_item)
                         rec_items_set.add(pop_item) # Обновляем множество для следующих итераций

        else:
            # --- Сценарий 2: Нет истории или нет похожих ---
            # Используем базовый подход: популярные, исключая купленные
            rec_items = [item for item in popular_items if item not in bought_by_user][:top_k]

        # Сохраняем рекомендации для пользователя
        recommendations[uid] = rec_items

    print(f"Рекомендации сгенерированы для {len(recommendations):,} пользователей")
    return recommendations

# --- Вызов функции ---
# Предполагается, что все необходимые переменные уже определены:
# popular_items, user_preferences, similar_users_dict, orders_df, test_users_during_period

# recommendations_hybrid = generate_recommendations_hybrid(
#     test_users=test_users_during_period,
#     popular_items=popular_items,
#     user_preferences=user_preferences,
#     similar_users_dict=similar_users_dict,
#     orders_df=orders_df, # Передаем весь датафрейм для построения индекса
#     top_k=100
# )

# --- Расчет метрики ---
# ndcg_score_hybrid = calculate_ndcg_batch(recommendations_hybrid, ground_truth, k=100)
# print(f"\n🎉 NDCG@100 для гибридной модели: {ndcg_score_hybrid:.6f}")


In [24]:
recommendations_hybrid = generate_recommendations_hybrid(
    test_users=test_users_during_period,
    popular_items=popular_items,
    user_preferences=user_preferences,
    similar_users_dict=similar_users_dict,
    orders_df=orders_df, # Передаем весь датафрейм для построения индекса
    top_k=100
)


=== 🎯 ГЕНЕРАЦИЯ ГИБРИДНЫХ РЕКОМЕНДАЦИЙ ===
Подготовка индекса покупок пользователей...
Индекс покупок создан для 805981 пользователей.


Генерация рекомендаций: 100%|██████████| 96159/96159 [00:03<00:00, 27207.28it/s]


Рекомендации сгенерированы для 96,159 пользователей


In [25]:
# --- Расчет метрики ---
ndcg_score_hybrid = calculate_ndcg_batch(recommendations_hybrid, ground_truth, k=100)
print(f"\n🎉 NDCG@100 для гибридной модели: {ndcg_score_hybrid:.6f}")


=== 📊 РАСЧЕТ МЕТРИКИ NDCG@100 ===


Расчет NDCG: 100%|██████████| 96159/96159 [00:01<00:00, 62318.57it/s]

Оценено пользователей: 96,159
NDCG@100: 0.018082

🎉 NDCG@100 для гибридной модели: 0.018082


In [26]:
# --- Расчет метрики для гибридной модели ---
ndcg_score_hybrid = calculate_ndcg_batch(recommendations_hybrid, ground_truth, k=100)

# === Вывод результатов ===
print("\n" + "="*50)
print("           СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*50)

# Старый результат (базовой модели) - используем переменную ndcg_score
ndcg_score_baseline = ndcg_score # <--- Берем из уже рассчитанной переменной
print(f"📉 Базовая модель (популярные товары): {ndcg_score_baseline:.6f}")

# Новый результат (гибридной модели)
print(f"🎉 Гибридная модель (CF + популярные) : {ndcg_score_hybrid:.6f}")

# Разница
improvement = ndcg_score_hybrid - ndcg_score_baseline
print("-" * 50)
print(f"📈 Разница (Гибрид - Базовая)         : {improvement:+.6f}")

# Интерпретация
if improvement > 0:
    print("✅ Гибридная модель лучше!")
elif improvement < 0:
    print("❌ Гибридная модель хуже базовой.")
else:
    print("🔄 Результаты моделей одинаковы.")
print("="*50)


=== 📊 РАСЧЕТ МЕТРИКИ NDCG@100 ===


Расчет NDCG: 100%|██████████| 96159/96159 [00:01<00:00, 62553.30it/s]


Оценено пользователей: 96,159
NDCG@100: 0.018082

           СРАВНЕНИЕ МОДЕЛЕЙ
📉 Базовая модель (популярные товары): 0.007641
🎉 Гибридная модель (CF + популярные) : 0.018082
--------------------------------------------------
📈 Разница (Гибрид - Базовая)         : +0.010441
✅ Гибридная модель лучше!
